# Recommendation System Using Collaborative Filtering and Matrix Factorization

This notebook builds a recommendation system for movie ratings. It includes:

- A sparse user-item ratings dataset
- Matrix factorization trained with stochastic gradient descent
- Item-based collaborative filtering as a baseline
- Evaluation with RMSE, MAE, Precision@5, and Recall@5
- Top-N recommendations for a sample user

## 1. Import Libraries

The implementation intentionally uses only `numpy` and `pandas`, so it can run in a lightweight environment.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 20)
pd.set_option('display.precision', 4)

## 2. Create a Ratings Dataset

The data below simulates users rating movies from different genres. This keeps the notebook offline and reproducible while still producing a realistic sparse recommendation problem.

In [2]:
RANDOM_SEED = 42

movies = pd.DataFrame([
    (1, 'The Space Between Stars', 'Sci-Fi'),
    (2, 'Laugh Track', 'Comedy'),
    (3, 'Midnight Evidence', 'Thriller'),
    (4, 'City of Letters', 'Drama'),
    (5, 'Orbit Academy', 'Sci-Fi'),
    (6, 'The Last Recipe', 'Drama'),
    (7, 'Weekend Chaos', 'Comedy'),
    (8, 'Silent Harbor', 'Thriller'),
    (9, 'Pixel Knights', 'Action'),
    (10, 'Hidden Signal', 'Sci-Fi'),
    (11, 'Rainy Day Promise', 'Romance'),
    (12, 'Courtroom Eleven', 'Drama'),
    (13, 'Fast Lane', 'Action'),
    (14, 'Island Hearts', 'Romance'),
    (15, 'The Puzzle Room', 'Thriller'),
    (16, 'Mountain Echo', 'Adventure'),
    (17, 'Desert Run', 'Adventure'),
    (18, 'Office Legends', 'Comedy'),
    (19, 'Neon Chase', 'Action'),
    (20, 'Northbound', 'Adventure'),
], columns=['movie_id', 'title', 'genre'])

def create_ratings(n_users=80, ratings_per_user=(8, 16), seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    genres = movies['genre'].unique()
    genre_to_idx = {genre: idx for idx, genre in enumerate(genres)}
    movie_quality = rng.normal(0.0, 0.45, len(movies))
    rows = []

    for user_id in range(1, n_users + 1):
        user_taste = rng.normal(0.0, 1.0, len(genres))
        user_bias = rng.normal(0.0, 0.35)
        watched_count = rng.integers(ratings_per_user[0], ratings_per_user[1] + 1)
        watched_movies = rng.choice(movies['movie_id'], size=watched_count, replace=False)

        for movie_id in watched_movies:
            movie = movies.loc[movies['movie_id'] == movie_id].iloc[0]
            genre_score = user_taste[genre_to_idx[movie['genre']]]
            noise = rng.normal(0.0, 0.45)
            rating = 3.2 + user_bias + movie_quality[movie_id - 1] + 0.75 * genre_score + noise
            rating = np.clip(np.round(rating * 2) / 2, 1.0, 5.0)
            rows.append((user_id, int(movie_id), float(rating)))

    return pd.DataFrame(rows, columns=['user_id', 'movie_id', 'rating'])

ratings = create_ratings()
ratings.head()

,user_id,movie_id,rating
0,1,12,4.0
1,1,20,4.0
2,1,15,4.0
3,1,2,2.5
4,1,7,3.0


In [3]:
summary = pd.DataFrame({
    'Metric': ['Users', 'Movies', 'Ratings', 'Sparsity'],
    'Value': [
        ratings['user_id'].nunique(),
        ratings['movie_id'].nunique(),
        len(ratings),
        f"{1 - len(ratings) / (ratings['user_id'].nunique() * ratings['movie_id'].nunique()):.2%}",
    ]
})
summary

,Metric,Value
0,Users,80
1,Movies,20
2,Ratings,935
3,Sparsity,41.56%


## 3. Train-Test Split

A user-wise split is used so each user's ratings are represented in both training and testing.

In [4]:
def train_test_split_by_user(ratings, test_fraction=0.2, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    test_indices = []

    for _, user_rows in ratings.groupby('user_id'):
        n_test = max(1, int(round(len(user_rows) * test_fraction)))
        test_indices.extend(rng.choice(user_rows.index, size=n_test, replace=False))

    test = ratings.loc[test_indices].copy()
    train = ratings.drop(test_indices).copy()
    return train.reset_index(drop=True), test.reset_index(drop=True)

train, test = train_test_split_by_user(ratings)
len(train), len(test)

(746, 189)

## 4. Matrix Factorization Model

The model learns latent vectors for users and movies. A predicted rating is calculated as:

`global_mean + user_bias + movie_bias + dot(user_vector, movie_vector)`

In [5]:
class MatrixFactorizationRecommender:
    def __init__(self, n_factors=12, n_epochs=80, learning_rate=0.03, reg=0.04, seed=RANDOM_SEED):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.learning_rate = learning_rate
        self.reg = reg
        self.seed = seed

    def fit(self, ratings):
        self.user_ids = sorted(ratings['user_id'].unique())
        self.movie_ids = sorted(ratings['movie_id'].unique())
        self.user_to_idx = {user_id: idx for idx, user_id in enumerate(self.user_ids)}
        self.movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(self.movie_ids)}
        self.global_mean = ratings['rating'].mean()

        rng = np.random.default_rng(self.seed)
        self.user_bias = np.zeros(len(self.user_ids))
        self.movie_bias = np.zeros(len(self.movie_ids))
        self.user_factors = rng.normal(0, 0.1, (len(self.user_ids), self.n_factors))
        self.movie_factors = rng.normal(0, 0.1, (len(self.movie_ids), self.n_factors))
        training_rows = ratings[['user_id', 'movie_id', 'rating']].to_numpy()

        for _ in range(self.n_epochs):
            rng.shuffle(training_rows)
            for user_id, movie_id, rating in training_rows:
                u = self.user_to_idx[int(user_id)]
                i = self.movie_to_idx[int(movie_id)]
                pred = self._predict_known(u, i)
                err = rating - pred
                old_user_factors = self.user_factors[u].copy()

                self.user_bias[u] += self.learning_rate * (err - self.reg * self.user_bias[u])
                self.movie_bias[i] += self.learning_rate * (err - self.reg * self.movie_bias[i])
                self.user_factors[u] += self.learning_rate * (err * self.movie_factors[i] - self.reg * self.user_factors[u])
                self.movie_factors[i] += self.learning_rate * (err * old_user_factors - self.reg * self.movie_factors[i])
        return self

    def _predict_known(self, user_idx, movie_idx):
        return self.global_mean + self.user_bias[user_idx] + self.movie_bias[movie_idx] + np.dot(self.user_factors[user_idx], self.movie_factors[movie_idx])

    def predict(self, user_id, movie_id):
        if user_id not in self.user_to_idx or movie_id not in self.movie_to_idx:
            return self.global_mean
        u = self.user_to_idx[user_id]
        i = self.movie_to_idx[movie_id]
        return float(np.clip(self._predict_known(u, i), 1.0, 5.0))

    def recommend(self, user_id, ratings, movies, top_n=5):
        rated_movie_ids = set(ratings.loc[ratings['user_id'] == user_id, 'movie_id'])
        candidates = movies.loc[~movies['movie_id'].isin(rated_movie_ids)].copy()
        candidates['predicted_rating'] = candidates['movie_id'].apply(lambda movie_id: self.predict(user_id, movie_id))
        return candidates.sort_values('predicted_rating', ascending=False).head(top_n)

model = MatrixFactorizationRecommender().fit(train)

## 5. Evaluation Metrics

- RMSE and MAE measure rating prediction error.
- Precision@5 and Recall@5 measure whether the top recommendations match movies the user actually liked in the test set.

In [6]:
def evaluate_rating_predictions(model, test):
    preds = np.array([model.predict(int(row.user_id), int(row.movie_id)) for row in test.itertuples()])
    actual = test['rating'].to_numpy()
    rmse = np.sqrt(np.mean((actual - preds) ** 2))
    mae = np.mean(np.abs(actual - preds))
    return {'RMSE': round(float(rmse), 4), 'MAE': round(float(mae), 4)}

def evaluate_top_k(model, train, test, k=5, relevant_threshold=4.0):
    precisions = []
    recalls = []

    for user_id in test['user_id'].unique():
        relevant = set(test.loc[(test['user_id'] == user_id) & (test['rating'] >= relevant_threshold), 'movie_id'])
        if not relevant:
            continue
        recommendations = model.recommend(int(user_id), train, movies, top_n=k)
        recommended = set(recommendations['movie_id'])
        hits = len(recommended & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))

    return {
        f'Precision@{k}': round(float(np.mean(precisions)), 4) if precisions else 0.0,
        f'Recall@{k}': round(float(np.mean(recalls)), 4) if recalls else 0.0,
    }

metrics = {**evaluate_rating_predictions(model, test), **evaluate_top_k(model, train, test, k=5)}
pd.DataFrame(metrics.items(), columns=['Metric', 'Value'])

,Metric,Value
0,RMSE,0.7761
1,MAE,0.6188
2,Precision@5,0.1660
3,Recall@5,0.6702


## 6. Recommendation Results

In [7]:
user_id = 7
model.recommend(user_id, train, movies, top_n=5)[['title', 'genre', 'predicted_rating']]

,title,genre,predicted_rating
3,City of Letters,Drama,4.2483
10,Rainy Day Promise,Romance,3.9155
14,The Puzzle Room,Thriller,3.7847
0,The Space Between Stars,Sci-Fi,3.7445
6,Weekend Chaos,Comedy,3.6267


## 7. Item-Based Collaborative Filtering Baseline

In [8]:
def item_similarity_recommendations(ratings, user_id, top_n=5):
    user_item = ratings.pivot_table(index='user_id', columns='movie_id', values='rating').fillna(0)
    matrix = user_item.to_numpy()
    norms = np.linalg.norm(matrix, axis=0, keepdims=True)
    normalized = np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms != 0)
    similarity = normalized.T @ normalized
    movie_columns = list(user_item.columns)

    rated = ratings.loc[ratings['user_id'] == user_id, ['movie_id', 'rating']]
    rated_ids = set(rated['movie_id'])
    scores = []

    for movie_id in movies['movie_id']:
        if movie_id in rated_ids:
            continue
        movie_idx = movie_columns.index(movie_id)
        score = 0.0
        sim_total = 0.0
        for row in rated.itertuples():
            if row.movie_id in movie_columns:
                rated_idx = movie_columns.index(row.movie_id)
                sim = similarity[movie_idx, rated_idx]
                score += sim * row.rating
                sim_total += abs(sim)
        scores.append((movie_id, score / sim_total if sim_total else 0.0))

    recs = pd.DataFrame(scores, columns=['movie_id', 'similarity_score'])
    return recs.merge(movies, on='movie_id').sort_values('similarity_score', ascending=False).head(top_n)

item_similarity_recommendations(train, user_id, top_n=5)[['title', 'genre', 'similarity_score']]

,title,genre,similarity_score
11,Office Legends,Comedy,3.9119
7,Rainy Day Promise,Romance,3.9050
9,The Puzzle Room,Thriller,3.9034
4,The Last Recipe,Drama,3.8762
6,Hidden Signal,Sci-Fi,3.8750


## Conclusion

The matrix factorization model learns hidden user preferences and movie characteristics from sparse rating data. The notebook reports both prediction quality and top-N recommendation quality, which directly satisfies the requirement to showcase recommendation results and evaluation metrics.